## LDN Generator

### Importing Libraries

In [1]:
import os
import shutil
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import cpu_count
from functools import partial
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

### Pathways of Folders

In [2]:
# ---------------- CONFIG ----------------
excel_path = r"C:\Users\chethan.m\Desktop\DLN Mar\Demand Notice Mar'26.xlsx"      # Path to your Excel file
output_folder = r"C:\Users\chethan.m\Desktop\DLN Mar\DLN"             # Folder to save PDFs
signature_path = r"C:\Users\chethan.m\Desktop\DLN Mar\Sign Of Axio (1).png" 
background_path = r"C:\Users\chethan.m\Desktop\DLN Mar\Page Background (1).png"
temp_output_folder = r"C:\Users\chethan.m\Desktop\DLN Mar\TEMP"
# Signature image path (must exist)

### Number of Batches (Adjust only Batch Size)

In [3]:
# Parallel & batching
MAX_WORKERS = max(1, cpu_count() - 1)
BATCH_SIZE = 10000

In [4]:
# Logging / failure
os.makedirs(output_folder, exist_ok=True)
os.makedirs(temp_output_folder, exist_ok=True)
logging.basicConfig(filename='pdf_generation.log', level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(message)s')
failure_log_path = r"C:\Users\praveen.kumar\Desktop\failed_rows.csv"

In [5]:
# ---------------- TEMPLATE ----------------
LETTER_BODY = """To,

<b>{NAME_BLOCK}</b>

<b>Sub : Your loan with axio (Earlier Capital Float) – Demand Notice</b>

Sir / Madam,
We, CapFloat Financial Services Private Limited operating under the brand name axio (Earlier Capital Float), and, having registered office at, “Gokaldas Platinum”, New No.3, (Old No.211), Bellary Road, Upper Palace Orchards, Sadashivanagar, Bengaluru – 560080, Karnataka, (hereinafter referred to as “the Company”) address you as hereunder: 
We wish to inform to you that, after duly complying with the requirements to submit the request for financial facility and on submitting all the requisite information along with copies of the Know Your Customer documents viz., PAN Card /Address Proof and execution of loan agreement, etc., we have extended the Financial Facility to you.

You the addressee above named have availed financial facility of Rs. <b>«Sanctioned_Amount»/</b>- vide Loan Account Number <b>«LAN»</b> from us, as per the terms of the loan agreement, you had agreed to pay EMI’s regularly without fail.

We observe that, in spite of repeated requests and reminders through SMS/ Phone Calls/E-Mails you have defaulted in paying the said EMI’s, and in lieu of the same your loan account has become overdue.
As a result thereof, the current outstanding is Rs. <b>«Outstanding_Amount»/</b>- as on inclusive of overdue / penalty charges. 

We state that the above violation is within your knowledge and non-regularization of the loan account is an event of default. In the event of failing to pay the amount mentioned in the Demand notice within 15 days, we shall be constrained to recall the loan facility which may have further implications including the credit bureau.
 
In view of the aforesaid facts and circumstances, we hereby call upon you to pay the outstanding sum of Rs. <b>«Outstanding_Amount»</b> as on and such further pending amount as on the date of realization to us immediately upon receipt of this Notice, through Online Payment/DD/Cheque. Kindly ignore this notice, if already paid the demanded amount.
If any queries please mail to ask@axio.co.in from your registered e-mail ID.



Yours Sincerely,

"""

# ---------------- PDF BACKGROUND FUNCTION ----------------
def add_background(canvas, doc):
    """Draws background image on each page."""
    if background_path and os.path.exists(background_path):
        from reportlab.lib.pagesizes import A4
        page_width, page_height = A4
        canvas.saveState()
        canvas.drawImage(background_path, 0, 0, width=page_width, height=page_height, preserveAspectRatio=True, mask='auto')
        canvas.restoreState()

# ---------------- WORKER FUNCTION ----------------
def generate_pdf_from_row(row_dict, temp_output_folder, signature_path):
    try:
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.enums import TA_JUSTIFY, TA_LEFT

        styles = getSampleStyleSheet()
        styles.add(ParagraphStyle(name='Justify', alignment=TA_JUSTIFY, leading=14))
        styles.add(ParagraphStyle(name='Left', alignment=TA_LEFT, leading=14))

        lan = str(row_dict.get("LAN", "")).strip()
        sl_no = str(row_dict.get("Sl_No", "")).strip()
        date_text = str(row_dict.get("Date", "")).strip()
        name = str(row_dict.get("Name_of_Borrower", "")).strip()
        sanctioned = str(row_dict.get("Sanction_Amount", "")).strip()
        outstanding = str(row_dict.get("Outstanding_Amount", "")).strip()

        name_block = name
        letter_text = LETTER_BODY.replace("«Sanctioned_Amount»", sanctioned)\
                                 .replace("«LAN»", lan)\
                                 .replace("«Outstanding_Amount»", outstanding)\
                                 .replace("{NAME_BLOCK}", name_block)

        filename = f"{lan}.pdf" if lan else f"row_{sl_no}.pdf"
        tmp_path = os.path.join(temp_output_folder, filename)

        doc = SimpleDocTemplate(tmp_path, pagesize=A4,
                                rightMargin=72, leftMargin=72,
                                topMargin=72, bottomMargin=72)
        story = []

        ref_no = f"Ref. No. {sl_no}"
        date_field = f"Date: {date_text}"
        table = Table([[ref_no, date_field]], colWidths=[doc.width * 0.6, doc.width * 0.4])
        table.setStyle(TableStyle([
            ('ALIGN', (0,0), (0,0), 'LEFT'),
            ('ALIGN', (1,0), (1,0), 'RIGHT'),
            ('FONTNAME', (0,0), (-1,-1), 'Helvetica'),
            ('FONTSIZE', (0,0), (-1,-1), 11),
            ('BOTTOMPADDING', (0,0), (-1,-1), 6),
            ('TOPPADDING', (0,0), (-1,-1), 6),
        ]))
        story.append(table)
        story.append(Spacer(1, 12))

        for para in letter_text.split("\n\n"):
            if para.strip():
                story.append(Paragraph(para.strip(), styles['Justify']))
                story.append(Spacer(1, 12))

        if signature_path and os.path.exists(signature_path):
            img = Image(signature_path, width=120, height=50, hAlign='LEFT')
            story.append(img)
            story.append(Spacer(1, 12))

        story.append(Paragraph(
            "Authorised Signatory<br/>CapFloat Financial Services Private Limited<br/>(Formerly Zen Lefin Private Limited)",
            styles['Left']
        ))

        # Add background on all pages
        doc.build(story, onFirstPage=add_background, onLaterPages=add_background)

        return True, lan, tmp_path
    except Exception as e:
        return False, str(row_dict.get("LAN", row_dict.get("Sl_No", "unknown"))), str(e)

# ---------------- BATCH RUNNER ----------------
def run_in_batches(df, temp_output_folder, output_folder, signature_path,
                   max_workers=MAX_WORKERS, batch_size=BATCH_SIZE):
    rows = df.to_dict(orient="records")
    failures = []

    for batch_start in range(0, len(rows), batch_size):
        batch = rows[batch_start: batch_start + batch_size]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(generate_pdf_from_row, row, temp_output_folder, signature_path): row for row in batch}

            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc=f"Batch {batch_start+1}-{min(batch_start+batch_size,len(rows))}"):
                row = futures[fut]
                try:
                    success, lan_or_id, result = fut.result()
                    if success:
                        final_path = os.path.join(output_folder, os.path.basename(result))
                        shutil.move(result, final_path)
                    else:
                        logging.error(f"Failed {lan_or_id}: {result}")
                        failures.append({**row, "error": result})
                except Exception as e:
                    logging.exception("Unhandled exception in future")
                    failures.append({**row, "error": str(e)})

    if failures:
        pd.DataFrame(failures).to_csv(failure_log_path, index=False)
        print(f"Completed with {len(failures)} failures. See {failure_log_path} and pdf_generation.log")
    else:
        print("Completed successfully with zero failures.")

# ---------------- MAIN ----------------
if __name__ == "__main__":
    df = pd.read_excel(excel_path)
    df.columns = [col.strip().replace(" ", "_").replace("-", "_") for col in df.columns]

    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.strftime("%d-%b-%Y").fillna('')

    os.makedirs(temp_output_folder, exist_ok=True)
    os.makedirs(output_folder, exist_ok=True)

    run_in_batches(df, temp_output_folder, output_folder, signature_path,
                   max_workers=MAX_WORKERS, batch_size=BATCH_SIZE)

Batch 10001-16029: 100%|███████████████████████████████████████████████████████████| 6029/6029 [46:47<00:00,  2.15it/s]


Completed successfully with zero failures.
